# 09 - Pandas: fundamentos — **Soluciones**

Soluciones a los ejercicios propuestos en el cuaderno 09.

## Contexto (recrear dataset del cuaderno original)

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "id": range(1, n+1),
    "edad": rng.integers(18, 70, size=n),
    "ingresos": rng.normal(25000, 8000, size=n).round(2),
    "ciudad": rng.choice(["Cádiz", "Sevilla", "Málaga", "Córdoba"], size=n),
    "compra": rng.choice([0,1], size=n, p=[0.4, 0.6]),
})
df = df.assign(
    tramo_edad = pd.cut(df["edad"], bins=[17,25,35,50,70], labels=["18-25","26-35","36-50","51-70"])
)
df.head()

## Ejercicio 1
**Crea `gasto_relativo = ingresos / ingresos.mean()`**

In [ ]:
ingresos_media = df["ingresos"].mean()
df["gasto_relativo"] = df["ingresos"] / ingresos_media
df[["ingresos","gasto_relativo"]].head()

## Ejercicio 2
**Tasa de compra por `tramo_edad` y ordenar de mayor a menor**

In [ ]:
tasa_por_tramo = (
    df.groupby("tramo_edad")["compra"]
      .mean()
      .sort_values(ascending=False)
      .rename("tasa_compra")
      .reset_index()
)
tasa_por_tramo

## Ejercicio 3
**Filtrar Sevilla con ingresos > 30000 y guardar a CSV**

In [ ]:
sevilla_altos = df[(df["ciudad"]=="Sevilla") & (df["ingresos"] > 30000)].copy()
out_path = "/mnt/data/09_sol_sevilla_ingresos_altos.csv"
sevilla_altos.to_csv(out_path, index=False)
out_path, sevilla_altos.head()

---

## ✅ Autotests 09

In [ ]:
# Pequeño helper de tests
def check(cond, msg_ok, msg_fail):
    if cond:
        print("✔", msg_ok)
    else:
        raise AssertionError("✘ " + msg_fail)

# Test 1: columnas esperadas
check(all(c in df.columns for c in ["edad","ingresos","ciudad","tramo_edad","gasto_relativo"]),
      "Columnas clave presentes (incluye 'tramo_edad' y 'gasto_relativo').",
      "Faltan columnas esperadas en df.")

# Test 2: gasto_relativo correcto (media ~ 1)
import numpy as np
mu = df["gasto_relativo"].mean()
check(abs(mu - 1.0) < 1e-6,
      "gasto_relativo tiene media ≈ 1.",
      f"Media de gasto_relativo inesperada: {mu}")

# Test 3: tasa_por_tramo descendente
check((tasa_por_tramo["tasa_compra"].values == sorted(tasa_por_tramo["tasa_compra"].values, reverse=True)).all(),
      "tasa_por_tramo está ordenado de mayor a menor.",
      "tasa_por_tramo no está ordenado descendentemente.")

# Test 4: CSV de Sevilla existe y cumple condiciones
from pathlib import Path
p = Path(out_path)
check(p.exists(), "CSV de Sevilla creado.", "No se encontró el CSV de Sevilla.")
tmp = pd.read_csv(p)
check(((tmp["ciudad"]=="Sevilla") & (tmp["ingresos"]>30000)).all() or len(tmp)==0,
      "Filtrado Sevilla > 30000 correcto (o vacío, válido según datos).",
      "El CSV contiene filas que no cumplen las condiciones.")